[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/16_embeddings_multimodales/16_embeddings_multimodales.ipynb)

# Módulo 16 · Embeddings multimodales

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **GPU** | L4 recomendada **solo** para las celdas opcionales con modelos reales (ColPali, CLIP/SigLIP); T4 vale con modelos pequeños. Todo lo demás corre en CPU con `FAST_DEV_RUN=True` (≈ 0 unidades de Colab; ≈ 5–8 min de CPU) |
| **Prerrequisitos** | 08 (tokens multimodales: patches ViT), 11 (InfoNCE), 13 (late interaction / MaxSim), 15 (evaluación) |
| **Hilo conductor** | Políglota Labs quiere que el buscador encuentre también **PDFs, capturas y diagramas** de la documentación |

> 🧭 **Ruta núcleo vs ampliación.** Núcleo (≈ 3 h): secciones 1–5 (CLIP, SigLIP, modality gap, ColPali en miniatura). Ampliación (≈ 2 h): 6–8 (embeddings unificados, audio/vídeo, decisión OCR vs visual).
> Las celdas con modelos reales están marcadas **(opcional, no probada en CPU)** y protegidas con flags.

## 🎯 Objetivos de aprendizaje
1. **Implementar** desde cero un modelo de dos torres con pérdida InfoNCE simétrica (CLIP) y con pérdida sigmoide (SigLIP) y **comparar** su comportamiento.
2. **Medir** el *modality gap* (distancia entre centroides), su dependencia de la temperatura y sus efectos en un índice de modalidades mezcladas; **aplicar** una corrección simple.
3. **Explicar y programar** MaxSim (late interaction) sobre *patch embeddings* de una página, y **diagnosticar** por qué un único vector medio falla.
4. **Comparar** un pipeline OCR + texto con un embedding visual según el tipo de consulta y el nivel de ruido, y **estimar** el coste de almacenamiento multi-vector.
5. **Situar** los embeddings unificados (Gemini Embedding 2, Jina v4, VLM2Vec) con sus capacidades **verificadas y fechadas**.
6. **Razonar** sobre audio y vídeo en espacios de embedding y sobre qué se pierde al promediar frames.
7. **Decidir** cuándo usar OCR+texto, embedding visual o ambos.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1.</b> (Módulo 08) Una imagen entra a un ViT como patches. Si la página tiene 448×448 px y los patches son de 14×14, ¿cuántos patches hay?</summary>

$(448/14)^2 = 32^2 = 1024$ patches. Es justo el orden de magnitud que usa ColPali (1024 patches por página con el codificador de visión de PaliGemma): **cada patch pasa a ser un vector**.
</details>

<details><summary><b>2.</b> (Módulo 11) Escribe la InfoNCE para un batch de $N$ pares y di qué son los negativos.</summary>

$\mathcal L=-\frac1N\sum_i\log\frac{e^{s_{ii}/\tau}}{\sum_j e^{s_{ij}/\tau}}$: los negativos son los otros $N-1$ elementos del batch (*in-batch negatives*). En CLIP se aplica en ambas direcciones (imagen→texto y texto→imagen).
</details>

<details><summary><b>3.</b> (Módulo 13) ¿Qué calcula MaxSim de ColBERT?</summary>

Para cada token de la query toma la **máxima** similitud con cualquier token del documento y suma: $\sum_i\max_j\langle q_i,d_j\rangle$. Aquí los «tokens del documento» serán los patches de la página.
</details>

<details><summary><b>4.</b> (Módulo 15) ¿Cómo comparas dos sistemas de retrieval con pocas queries?</summary>

Métricas por query + bootstrap/test pareado con IC (nunca dos medias sueltas). Lo usaremos en el proyecto con `poliglota_eval.py`.
</details>

## 🗺️ Mapa de madurez (estado a oct-2026)

| Técnica | Madurez | Quién / dónde | Fuente |
|---|---|---|---|
| CLIP (contraste imagen-texto con InfoNCE) | 🟢 sólido | Base de casi todo el ecosistema texto-imagen | Radford et al. 2021 |
| SigLIP (pérdida sigmoide) y SigLIP 2 (multilingüe, captioning + auto-distilación + datos curados) | 🟢 sólido / 🟡 SigLIP 2 en adopción | Google; codificadores de visión de PaliGemma | Zhai et al. 2023; Tschannen et al. 2025 |
| Modality gap: existencia y medición | 🟢 sólido (fenómeno); 🟡 cuánto importa en producto | Investigación; práctica de índices mixtos | Liang et al. 2022 |
| Retrieval visual de documentos con multi-vector (ColPali / ColQwen) | 🟡 emergente (resultados fuertes en ViDoRe; coste de almacenamiento) | ILLUIN y comunidad; también Jina v4 (dense + multi-vector) | Faysse et al. 2024; Günther et al. 2025 |
| Un único vector por captura de pantalla (DSE) | 🟡 emergente | Investigación (Waterloo) | Ma et al. 2024 |
| Embeddings unificados multimodales (texto, imagen, vídeo, audio, PDF en un espacio) | 🟡 emergente: Gemini Embedding 2 (preview mar-2026, GA abr-2026 según prensa) | Google; también Jina v4, VLM2Vec en investigación | ver §6 |
| Espacios de audio/vídeo (CLAP, ImageBind) | 🟡 / 🔴 | Investigación | Elizalde 2022; Girdhar 2023 |
| Embeddings de vídeo con orden temporal fino | 🔴 experimental | — | (§7) |

## 1. Intuición: dos torres, un espacio

Un **embedding multimodal** coloca contenidos de distinto tipo (una foto y su descripción) **cerca** en el mismo espacio vectorial, de modo que se pueda buscar «foto de un círculo rojo» y obtener la imagen. Es la misma idea que el *two-tower* de recomendación (usuario ↔ ítem) que viste en `recsys-course`: dos codificadores, un producto escalar, y entrenamiento con negativos del batch.

```
imagen ──► [torre de imagen] ──► z_i ─┐
                                      ├─ coseno ──► InfoNCE (diagonal = pares correctos)
texto  ──► [torre de texto ] ──► z_t ─┘
```
Para poder **medir** todo en CPU y en minutos construiremos un CLIP en miniatura con figuras de colores y captions en **español e inglés** (la imagen hace de «pivote» entre los dos idiomas). Después subiremos de escala con modelos reales (celdas opcionales).

> ### 🎨 GEMINI-IMAGEN · ID: `M16-IMG-1` · ⚠️ REEMPLAZAR ESTA CELDA
> **CLIP y SigLIP: dos torres y la matriz de similitudes**
>
> **Qué muestra:** Arquitectura de dos torres (imagen y texto), la matriz N×N de similitudes y la diferencia entre InfoNCE (softmax por fila y columna) y SigLIP (sigmoide por celda).
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Dos torres (imagen y texto) producen vectores normalizados cuya matriz de similitudes se entrena con InfoNCE (softmax) o con la pérdida sigmoide de SigLIP.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Composición con tres zonas de izquierda a derecha.
> Zona izquierda: dos columnas verticales de bloques. Arriba una caja azul «Torre de imagen (ViT / CNN)» con una miniatura abstracta de un círculo rojo a su izquierda; abajo una caja naranja «Torre de texto (Transformer)» con el texto «un círculo rojo» a su izquierda. De cada torre sale una flecha hacia un vector etiquetado «z_i» (azul) y «z_t» (naranja), ambos con la etiqueta «L2-normalizado».
> Zona central: una cuadrícula 4×4 con filas etiquetadas «imagen 1…4» y columnas «texto 1…4»; la diagonal en verde con «s_ii (par correcto)» y el resto en gris claro con «s_ij (negativos)». Título: «Matriz de similitudes N×N».
> Zona derecha: dos recuadros apilados. Recuadro superior (azul) «CLIP · InfoNCE»: texto «softmax sobre cada fila y cada columna» y la fórmula «L = −log( exp(s_ii/τ) / Σ_j exp(s_ij/τ) )». Recuadro inferior (verde) «SigLIP · sigmoide»: texto «cada celda es una clasificación binaria independiente (+1 diagonal, −1 fuera)» y la fórmula «L = −Σ_ij log σ( z_ij · (t·s_ij + b) )».
> Título superior: «Dos torres, un espacio de embeddings».
> Evita: texto ilegible o inventado, fotografías realistas, sombras 3D, elementos decorativos, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M16-IMG-1.png` en la carpeta del módulo y reemplaza esta celda por: `![Dos torres (imagen y texto) producen vectores normalizados cuya matriz de similitudes se entrena con InfoNCE (softmax) o con la pérdida sigmoide de SigLIP.](assets/M16-IMG-1.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> imagen -> [torre imagen] -> z_i --\
>                                   >-- S = z_i z_t^T (N x N) -- CLIP: softmax fila/col ; SigLIP: sigmoide por celda
> texto  -> [torre texto ] -> z_t --/       diagonal = pares correctos
> ```

> 👀 **Qué debes observar.** En CLIP los negativos compiten entre sí (softmax): el valor de cada celda depende del resto del batch. En SigLIP cada celda es independiente (sigmoide), por lo que **no hace falta normalizar sobre todo el batch** y se simplifica el entrenamiento distribuido (Zhai et al., 2023).

## 2. Entorno y datos de juguete

In [ ]:
!pip install -q numpy scikit-learn matplotlib pandas torch pillow transformers datasets

In [ ]:
import os, math, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42; FAST_DEV_RUN = True
random.seed(SEED); np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
if FAST_DEV_RUN: torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · semilla={SEED}")

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

FORMAS = ["circle", "square", "triangle", "cross"]; COLORES = ["red", "green", "blue", "yellow"]
RGB = {"red": (.9, .1, .1), "green": (.1, .7, .1), "blue": (.1, .2, .9), "yellow": (.95, .85, .1)}
ES_F = {"circle": ("un", "círculo"), "square": ("un", "cuadrado"), "triangle": ("un", "triángulo"), "cross": ("una", "cruz")}
ES_C = {"red": ("rojo", "roja"), "green": ("verde", "verde"), "blue": ("azul", "azul"), "yellow": ("amarillo", "amarilla")}
CONCEPTOS = [(f, c) for f in FORMAS for c in COLORES]          # 16 conceptos

def dibuja(forma: str, color: str, rng, S: int = 32, centro=None) -> np.ndarray:
    """Imagen SxSx3 float32 con una figura de color sobre fondo gris con ruido. Posición y tamaño aleatorios."""
    y, x = np.mgrid[0:S, 0:S]; cx, cy = rng.integers(int(S*.34), int(S*.66), 2) if centro is None else centro; r = rng.integers(max(2, int(S*.19)), int(S*.31))
    if forma == "circle": m = (x - cx)**2 + (y - cy)**2 < r * r
    elif forma == "square": m = (abs(x - cx) < r * .8) & (abs(y - cy) < r * .8)
    elif forma == "triangle": m = (y >= cy - r) & (y <= cy + r) & (abs(x - cx) <= (y - (cy - r)) / 2 + .5)
    else: m = ((abs(x - cx) < max(1, r / 3)) & (abs(y - cy) < r)) | ((abs(y - cy) < max(1, r / 3)) & (abs(x - cx) < r))
    img = np.full((S, S, 3), .5) + rng.normal(0, .04, (S, S, 3)); img[m] = np.array(RGB[color]) + rng.normal(0, .04, (m.sum(), 3))
    return np.clip(img, 0, 1).astype(np.float32)

def caption(forma: str, color: str, lang: str, rng) -> list:
    """Caption como lista de palabras, en ES o EN, con 3 plantillas por idioma (concordancia de género en ES)."""
    if lang == "en":
        return rng.choice(["a {c} {s}", "picture of a {c} {s}", "a photo of a {c} {s}"]).format(c=color, s=forma).split()
    art, s = ES_F[forma]; col = ES_C[color][1 if art == "una" else 0]
    return rng.choice(["{a} {s} {c}", "imagen de {a} {s} {c}", "una foto de {a} {s} {c}"]).format(a=art, s=s, c=col).split()

def make_dataset(n_per: int, seed: int):
    """Devuelve (imágenes, captions, etiqueta de concepto, idioma). Cada imagen lleva UNA caption (ES o EN al azar)."""
    rng = np.random.default_rng(seed); X, T, Y, L = [], [], [], []
    for k, (f, c) in enumerate(CONCEPTOS):
        for _ in range(n_per):
            lang = "es" if rng.random() < .5 else "en"
            X.append(dibuja(f, c, rng)); T.append(caption(f, c, lang, rng)); Y.append(k); L.append(lang)
    return np.stack(X), T, np.array(Y), L

In [ ]:
Xtr, Ttr, Ytr, Ltr = make_dataset(80, seed=1); Xte, Tte, Yte, Lte = make_dataset(20, seed=2)
vocab = sorted({w for t in Ttr + Tte for w in t}); w2i = {w: i + 1 for i, w in enumerate(vocab)}
print(f"{len(Xtr)} imágenes de entrenamiento · {len(Xte)} de test · vocabulario de captions: {len(vocab)} palabras (ES+EN)")
fig, ax = plt.subplots(2, 6, figsize=(12, 4.4))
for a, k in zip(ax.ravel(), np.random.default_rng(0).choice(len(Xte), 12, replace=False)):
    a.imshow(Xte[k]); a.set_title(" ".join(Tte[k])[:26], fontsize=8); a.axis("off")
fig.suptitle("Dataset de juguete: 16 conceptos (forma × color), captions en ES o EN", y=1.02); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Cada imagen tiene una sola caption, **en español o en inglés al azar**. El modelo nunca ve «un círculo rojo» y «a red circle» juntos: solo se conectarán a través de la **imagen** que ambos describen (pivote visual).
> Con 16 conceptos y batches de 128 habrá muchos *falsos negativos* en el batch (otra imagen del mismo concepto): el suelo teórico de la pérdida no es 0 (≈ $\ln(128/16)\approx2.1$).

## 3. CLIP en miniatura desde cero
**Pérdida (InfoNCE simétrica).** Con $z_i,z_t$ normalizados y escala $1/\tau$:
$$\mathcal L=\tfrac12\Big[\mathrm{CE}\big(\tfrac{1}{\tau}Z_iZ_t^\top,\ \mathrm{diag}\big)+\mathrm{CE}\big(\tfrac{1}{\tau}Z_tZ_i^\top,\ \mathrm{diag}\big)\Big]$$
$\tau$ es la **temperatura**: pequeña ⇒ distribución muy picuda (castiga mucho al negativo más cercano); grande ⇒ casi uniforme. En CLIP se **aprende** (inicializada en 0.07).

**SigLIP.** Sustituye el softmax por una sigmoide por par, con escala $t$ y sesgo $b$ aprendibles (el paper inicializa $t'=\log 10$, $b=-10$):
$$\mathcal L=-\frac1N\sum_{i}\sum_{j}\log\sigma\big(y_{ij}\,(t\,z_i\!\cdot\! z_j+b)\big),\quad y_{ij}=\begin{cases}+1&i=j\\-1&i\ne j\end{cases}$$
El sesgo $b\approx-10$ compensa que hay $N^2-N$ negativos frente a $N$ positivos desde el primer paso.

In [ ]:
class TorreImagen(nn.Module):
    """CNN diminuta -> vector L2-normalizado (el análogo de un ViT en miniatura)."""
    def __init__(self, D: int = 32):
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, D))
    def forward(self, x): return F.normalize(self.c(x.permute(0, 3, 1, 2)), dim=-1)

class TorreTexto(nn.Module):
    """Bolsa de palabras (media de embeddings) + capa lineal -> vector L2-normalizado."""
    def __init__(self, V: int, D: int = 32):
        super().__init__(); self.e = nn.EmbeddingBag(V + 1, D, mode="mean"); self.l = nn.Linear(D, D)
    def forward(self, ids, off): return F.normalize(self.l(self.e(ids, off)), dim=-1)

def codifica_textos(T, w2i):
    ids = [torch.tensor([w2i[w] for w in t]) for t in T]
    return torch.cat(ids), torch.tensor([0] + [len(i) for i in ids[:-1]]).cumsum(0)

def perdida_clip(zi, zt, escala):
    """InfoNCE simétrica: entropía cruzada fila a fila y columna a columna sobre la matriz N x N de logits."""
    lg = escala * zi @ zt.T; lab = torch.arange(len(zi))
    return (F.cross_entropy(lg, lab) + F.cross_entropy(lg.T, lab)) / 2

def perdida_siglip(zi, zt, escala, sesgo):
    """Sigmoide por PAR (sin softmax): +1 en la diagonal, -1 fuera. Suma sobre todos los pares / N (Zhai et al. 2023)."""
    lg = escala * zi @ zt.T + sesgo; lab = 2 * torch.eye(len(zi)) - 1
    return -F.logsigmoid(lab * lg).sum() / len(zi)

def entrena_clip(Xtr, Ttr, w2i, tipo="infonce", tau=None, epochs=60, bs=128, seed=0, D=32, lr=3e-3):
    """tau=None -> temperatura APRENDIDA (init 1/0.07 en infonce, t=10 y b=-10 en sigmoid). tau=float -> temperatura FIJA."""
    torch.manual_seed(seed); I, Tx = TorreImagen(D), TorreTexto(len(w2i), D)
    logt = nn.Parameter(torch.tensor(float(np.log(1 / 0.07 if tipo == "infonce" else 10.0)))); b = nn.Parameter(torch.tensor(-10.0))
    params = list(I.parameters()) + list(Tx.parameters()) + ([logt] if tau is None else []) + ([b] if tipo == "sigmoid" else [])
    opt = torch.optim.Adam(params, lr=lr); X = torch.tensor(Xtr); n = len(X); hist = []
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n - bs + 1, bs):
            idx = perm[i:i + bs]; ids, off = codifica_textos([Ttr[j] for j in idx], w2i); zi, zt = I(X[idx]), Tx(ids, off)
            escala = logt.exp() if tau is None else torch.tensor(1.0 / tau)
            loss = perdida_clip(zi, zt, escala) if tipo == "infonce" else perdida_siglip(zi, zt, escala, b)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(loss.item())
    return I, Tx, hist

@torch.no_grad()
def embebe(I, Tx, X, T, w2i):
    ids, off = codifica_textos(T, w2i); return I(torch.tensor(X)), Tx(ids, off)

def modality_gap(zi, zt) -> float:
    """Distancia entre los centroides de las dos modalidades (Liang et al. 2022), con embeddings L2-normalizados."""
    return float((zi.mean(0) - zt.mean(0)).norm())

def metricas_clip(zi, zt, Y) -> dict:
    """Precisión imagen->concepto del texto más cercano, R@1 texto->imagen (mismo concepto) y gap."""
    Y = torch.as_tensor(Y); S = zi @ zt.T
    return {"acc_i2t": float((Y[S.argmax(1)] == Y).float().mean()), "acc_t2i": float((Y[S.argmax(0)] == Y).float().mean()), "gap": modality_gap(zi, zt)}

In [ ]:
CONFIGS = [("infonce", None, "InfoNCE τ aprendida"), ("infonce", 0.01, "InfoNCE τ=0.01"), ("infonce", 0.05, "InfoNCE τ=0.05"),
           ("infonce", 0.2, "InfoNCE τ=0.2"), ("infonce", 1.0, "InfoNCE τ=1.0"), ("sigmoid", None, "SigLIP (sigmoide)")]
EPOCHS = 60 if FAST_DEV_RUN else 120
MODELOS, t0 = {}, time.time()
for tipo, tau, nombre in CONFIGS:
    I, Tx, hist = entrena_clip(Xtr, Ttr, w2i, tipo=tipo, tau=tau, epochs=EPOCHS)
    zi, zt = embebe(I, Tx, Xte, Tte, w2i); MODELOS[nombre] = dict(I=I, Tx=Tx, hist=hist, zi=zi, zt=zt, **metricas_clip(zi, zt, Yte))
    print(f"{nombre:22s} acc i→t={MODELOS[nombre]['acc_i2t']:.2f}  acc t→i={MODELOS[nombre]['acc_t2i']:.2f}  gap={MODELOS[nombre]['gap']:.3f}  ({time.time()-t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8)); nombres = list(MODELOS)
for n in nombres: ax[0].plot(MODELOS[n]["hist"], label=n)
ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida (último batch)"); ax[0].set_title("Curvas de entrenamiento"); ax[0].legend(frameon=False, fontsize=7)
ax[1].bar(range(len(nombres)), [MODELOS[n]["acc_i2t"] for n in nombres], color=[AZUL] * 5 + [VERDE]); ax[1].set_xticks(range(len(nombres))); ax[1].set_xticklabels(nombres, rotation=30, ha="right", fontsize=7)
ax[1].set_ylabel("precisión imagen → concepto"); ax[1].set_title("Calidad del retrieval cruzado")
ax[2].bar(range(len(nombres)), [MODELOS[n]["gap"] for n in nombres], color=[NARANJA] * 5 + [VERDE]); ax[2].set_xticks(range(len(nombres))); ax[2].set_xticklabels(nombres, rotation=30, ha="right", fontsize=7)
ax[2].set_ylabel("modality gap (distancia entre centroides)"); ax[2].set_title("Modality gap por configuración")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) La calidad se degrada si $\tau$ es demasiado grande (1.0): la señal contrastiva se diluye. (2) El **gap cae de forma monótona al subir $\tau$**, y solo ahí podría «cerrarse» a costa de calidad: no es gratis. (3) SigLIP aprende también, con un gap intermedio.
> Los números exactos dependen de semilla y épocas; lo robusto es la **tendencia**. Nota: en la sección siguiente se compara contra el gap del modelo *sin entrenar*.

## 4. SigLIP 2 y el panorama de modelos reales
**SigLIP 2** (Tschannen et al., 2025) mantiene el objetivo image-text de SigLIP y añade pretraining con *captioning*, pérdidas auto-supervisadas (auto-distilación y *masked prediction*), curación de datos online, una mezcla multilingüe menos sesgada y variantes con distintas resoluciones y relación de aspecto nativa.
En producción conviene probar **ambos**: SigLIP/SigLIP 2 suelen ser mejores en zero-shot que CLIP del mismo tamaño en los benchmarks de sus papers, pero la ganancia en *tu* dominio se mide, no se supone (módulo 15).

**(opcional, no probada en CPU)** Con red y GPU puedes repetir la medición del gap con modelos reales sobre nuestras figuras: pon `USE_REAL_MODELS = True`.

In [ ]:
USE_REAL_MODELS = False     # True en Colab con GPU/red (descarga ~0.6-1.5 GB). Probado solo en CPU con el juguete; esta celda NO se ejecutó en el entorno de desarrollo.
if USE_REAL_MODELS:
    from transformers import AutoModel, AutoProcessor
    from PIL import Image
    for nombre in ("openai/clip-vit-base-patch32", "google/siglip-base-patch16-224"):
        try:
            m = AutoModel.from_pretrained(nombre).to(device).eval(); pr = AutoProcessor.from_pretrained(nombre)
            imgs = [Image.fromarray((x * 255).astype("uint8")).resize((224, 224)) for x in Xte[:64]]; txt = [" ".join(t) for t in Tte[:64]]
            with torch.no_grad():
                ei = m.get_image_features(**pr(images=imgs, return_tensors="pt").to(device)); et = m.get_text_features(**pr(text=txt, padding="max_length", return_tensors="pt").to(device))
            ei, et = torch.nn.functional.normalize(ei, dim=-1).cpu(), torch.nn.functional.normalize(et, dim=-1).cpu()
            print(nombre, "modality gap =", round(modality_gap(ei, et), 3))
        except Exception as e:
            print(nombre, "no disponible:", type(e).__name__, str(e)[:80])
else:
    print("USE_REAL_MODELS=False: celda omitida.")

## 5. El modality gap, medido
**Liang et al. (NeurIPS 2022)** observaron que en CLIP y otros modelos multimodales las imágenes y los textos ocupan **regiones separadas** de la hiperesfera (a «un brazo de distancia»). Dos causas:
1. **Efecto cono** *(inicialización)*: las representaciones de una red profunda ocupan un cono estrecho; con dos redes distintas, **dos conos distintos** desde el paso 0.
2. **Aprendizaje contrastivo**: con temperatura baja la optimización *conserva* esa separación; la temperatura modifica su tamaño.

**Cómo se mide:** $\text{gap}=\lVert\bar z_i-\bar z_t\rVert_2$ con $\bar z$ el centroide de cada modalidad (embeddings normalizados). Medimos primero el modelo **sin entrenar** y comparamos.

In [ ]:
torch.manual_seed(0); I0, T0 = TorreImagen(), TorreTexto(len(w2i)); zi0, zt0 = embebe(I0, T0, Xte, Tte, w2i)
coseno_intra = lambda z: float((z @ z.T).mean())
filas = [{"modelo": "sin entrenar", "gap": modality_gap(zi0, zt0), "coseno medio intra-imagen": coseno_intra(zi0), "coseno medio intra-texto": coseno_intra(zt0)}]
for n in ("InfoNCE τ=0.01", "InfoNCE τ aprendida", "InfoNCE τ=0.2"):
    filas.append({"modelo": n, "gap": MODELOS[n]["gap"], "coseno medio intra-imagen": coseno_intra(MODELOS[n]["zi"]), "coseno medio intra-texto": coseno_intra(MODELOS[n]["zt"])})
pd.DataFrame(filas).set_index("modelo").round(3)

> 👀 **Qué debes observar.** Sin entrenar, las imágenes están **casi todas en el mismo punto** (coseno medio intra-imagen ≈ 1: efecto cono extremo) y a distancia grande de los textos. El entrenamiento reparte cada modalidad por la esfera (coseno intra baja) pero **el gap no desaparece** salvo con temperaturas altas.

In [ ]:
def pca2(Z, ref):
    c = ref.mean(0); _, _, Vt = np.linalg.svd((ref - c).numpy(), full_matrices=False); return (Z - c).numpy() @ Vt[:2].T
escenas = [("sin entrenar", zi0, zt0), ("InfoNCE τ=0.01", MODELOS["InfoNCE τ=0.01"]["zi"], MODELOS["InfoNCE τ=0.01"]["zt"]), ("InfoNCE τ=0.2", MODELOS["InfoNCE τ=0.2"]["zi"], MODELOS["InfoNCE τ=0.2"]["zt"])]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, (n, zi, zt) in zip(ax, escenas):
    ref = torch.cat([zi, zt]); pi, pt = pca2(zi, ref), pca2(zt, ref)
    a.scatter(*pi.T, s=8, color=AZUL, alpha=.6, label="imágenes"); a.scatter(*pt.T, s=8, color=NARANJA, alpha=.6, label="textos")
    a.set_title(f"{n} · gap={modality_gap(zi, zt):.2f}", fontsize=10); a.set_xlabel("PC1"); a.set_ylabel("PC2"); a.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** En el primer panel hay **dos nubes totalmente separadas** (los dos conos). Con $\tau=0.01$ siguen separadas (gap grande); con $\tau=0.2$ se solapan. Es la figura clásica de Liang et al. reproducida con tu propio modelo.

> ### 🎨 GEMINI-IMAGEN · ID: `M16-IMG-2` · ⚠️ REEMPLAZAR ESTA CELDA
> **El modality gap y el efecto cono**
>
> **Qué muestra:** Dos conos estrechos en la hiperesfera (imágenes y textos) separados por el vector del gap; consecuencia: en un índice mixto un texto tiene más cerca otros textos que su imagen correcta.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Dos conos en la hiperesfera (imágenes y textos) separados por el vector del gap y su efecto en un índice mixto.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Composición con dos paneles.
> Panel izquierdo, título «Efecto cono: dos redes, dos conos»: una esfera dibujada como círculo gris con dos zonas en forma de cono estrecho: una azul etiquetada «imágenes» (arriba a la izquierda) y una naranja etiquetada «textos» (abajo a la derecha). Entre los centroides (puntos grandes) una flecha negra etiquetada «vector del gap  Δ = z̄_img − z̄_txt» y la medida «gap = ‖Δ‖».
> Panel derecho, título «Consecuencia en un índice mixto»: la misma disposición, con una consulta de texto «un círculo rojo» (estrella naranja) dentro del cono naranja; tres puntos naranjas vecinos (otros textos) a distancia corta etiquetados «textos más cercanos», y un punto azul (la imagen correcta) a distancia mayor etiquetado «su imagen (más lejos)». Debajo, el texto: «Con gap grande, el top-k mezcla poco las modalidades».
> Abajo un recuadro verde: «Mitigación simple: restar el centroide de cada modalidad y renormalizar».
> Evita: texto ilegible o inventado, ilustraciones decorativas, sombras 3D, más de dos conos, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M16-IMG-2.png` en la carpeta del módulo y reemplaza esta celda por: `![Dos conos en la hiperesfera (imágenes y textos) separados por el vector del gap y su efecto en un índice mixto.](assets/M16-IMG-2.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> (imágenes)  ●─────── Δ (gap) ───────►  ●  (textos)
> una consulta de texto ★ tiene más cerca otros textos que su imagen correcta
> mitigación: restar el centroide de cada modalidad y renormalizar
> ```

> 👀 **Qué debes observar.** El gap **no importa** si solo comparas texto→imágenes (ranking dentro de una modalidad). Importa cuando **mezclas modalidades en el mismo índice** o comparas puntuaciones entre modalidades: el vecino más cercano de un texto tiende a ser otro texto. Lo medimos a continuación con la «tasa de intrusos».

In [ ]:
def centra(z, mu): return torch.nn.functional.normalize(z - mu, dim=-1)

def tasa_intruso(zi, zt, zq, Y, Yq):
    """Índice MIXTO (imágenes + captions de test). Para cada caption-query NUEVA: ¿hay un texto de OTRO concepto más cercano
    que la mejor imagen del concepto correcto? Devuelve (tasa de 'intrusos', margen medio imagen-correcta menos intruso)."""
    Y, Yq = torch.as_tensor(Y), torch.as_tensor(Yq); si, st = zq @ zi.T, zq @ zt.T
    mejor_img = torch.where(Y[None, :] == Yq[:, None], si, torch.full_like(si, -9)).max(1).values
    mejor_txt_mal = torch.where(Y[None, :] != Yq[:, None], st, torch.full_like(st, -9)).max(1).values
    return float((mejor_txt_mal > mejor_img).float().mean()), float((mejor_img - mejor_txt_mal).mean())

Xq, Tq_, Yq, Lq = make_dataset(10, seed=3)
filas = []
for n in ("InfoNCE τ=0.01", "InfoNCE τ aprendida", "InfoNCE τ=0.2", "SigLIP (sigmoide)"):
    M = MODELOS[n]; _, zq = embebe(M["I"], M["Tx"], Xq[:1], Tq_, w2i)
    r0, m0 = tasa_intruso(M["zi"], M["zt"], zq, Yte, Yq); mi, mt = M["zi"].mean(0), M["zt"].mean(0)
    r1, m1 = tasa_intruso(centra(M["zi"], mi), centra(M["zt"], mt), centra(zq, mt), Yte, Yq)
    filas.append({"modelo": n, "gap": M["gap"], "intrusos (original)": r0, "intrusos (centrado)": r1, "margen original": m0, "margen centrado": m1})
sesgo = pd.DataFrame(filas).set_index("modelo"); display(sesgo.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4)); x = np.arange(len(sesgo)); w = .38
ax[0].bar(x - w/2, sesgo["intrusos (original)"], w, color=ROJO, label="original"); ax[0].bar(x + w/2, sesgo["intrusos (centrado)"], w, color=VERDE, label="centrado por modalidad")
ax[0].set_xticks(x); ax[0].set_xticklabels(sesgo.index, rotation=15, fontsize=8); ax[0].set_ylabel("tasa de 'intrusos' (texto equivocado > imagen correcta)"); ax[0].legend(frameon=False, fontsize=8); ax[0].set_title("Índice mixto: texto equivocado que supera a la imagen correcta")
ax[1].scatter(sesgo["gap"], sesgo["intrusos (original)"], s=70, color=AZUL); [ax[1].annotate(n, (g, r), fontsize=7, xytext=(4, 4), textcoords="offset points") for n, g, r in zip(sesgo.index, sesgo["gap"], sesgo["intrusos (original)"])]
ax[1].set_xlabel("modality gap"); ax[1].set_ylabel("tasa de intrusos"); ax[1].set_title("Más gap, más intrusos")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con un gap pequeño o moderado (≤ 0.35 aquí) **no hay intrusos**: el texto correcto→imagen correcta sigue ganando a cualquier texto equivocado. Con $\tau=0.01$ (gap ≈ 1) **el 100 % de las queries tiene un texto equivocado por delante de la imagen correcta**: el índice mixto es inservible aunque el retrieval texto→imagen *por separado* funcione.
> Y ojo: **restar el centroide de cada modalidad no lo arregla** en el caso extremo (las imágenes siguen en un cono estrecho: coseno intra-imagen 0.62). La corrección por media es un parche, no un cambio de geometría; en la práctica se opta por índices separados con fusión por rangos (módulo 13) o por un modelo entrenado para un espacio unificado.

In [ ]:
# El pivote visual: ES ↔ EN sin pares ES-EN directos. Para cada caption ES de test, ¿su vecino EN más cercano es del mismo concepto?
M = MODELOS["InfoNCE τ aprendida"]; zt = M["zt"]; Yt = torch.as_tensor(Yte); es = torch.tensor([l == "es" for l in Lte]); en = ~es
S = zt[es] @ zt[en].T; acierto = float((Yt[en][S.argmax(1)] == Yt[es]).float().mean())
print(f"caption ES → caption EN del mismo concepto (top-1): {acierto:.2f}  (azar = {1/16:.2f}) · entrenamiento SIN pares ES-EN directos")

> 👀 **Qué debes observar.** El alineamiento cross-lingüe aparece **gratis**: las dos lenguas se conectan por la imagen. Es la intuición de por qué CLIP multilingüe (y SigLIP 2) pueden dar buena búsqueda texto→imagen en idiomas con pocas parejas texto-texto. 🟡 En modelos reales la calidad por idioma es muy desigual: evalúala con tus queries en español.

## 6. Retrieval visual de documentos: de OCR + texto a ColPali

**El problema.** Una documentación técnica real contiene PDFs con tablas, diagramas, capturas de pantalla y gráficos. El pipeline clásico es: *parsear/OCR → trocear → embeber texto → indexar*. Cada paso pierde información (layout, tablas, figuras) y es caro de mantener.

**Tres familias** (Faysse et al., 2024; Ma et al., 2024):
| Enfoque | Qué indexa por página | Pros | Contras |
|---|---|---|---|
| OCR + texto + embedder de texto | texto extraído (a veces + captions de figuras) | barato de almacenar, maduro | falla con layout/figuras; calidad = calidad del OCR |
| **DSE** (*Document Screenshot Embedding*): una captura → **un vector** | 1 vector por página | índice pequeño | un vector comprime todo |
| **ColPali / ColQwen**: la página como imagen → un VLM → **un vector por patch** + late interaction | ≈ 1 000 vectores de 128-d | recupera detalles locales, sin OCR | almacenamiento y cómputo de MaxSim |

**ColPali** (Faysse et al., ICLR 2025) usa PaliGemma-3B (SigLIP-So400m + Gemma-2B): 1 024 patches de la imagen + 6 tokens de texto, proyectados a $D=128$; entrena con una pérdida contrastiva *late interaction* sobre 118 695 pares query–página. En **ViDoRe** obtiene 81.3 de nDCG@5 medio frente a 67.0 del mejor pipeline de texto del paper (Unstructured + captioning + BGE-M3), e indexa una página en 0.39 s frente a 7.22 s del pipeline completo (GPU L4; Tablas 2 y 5 del paper). ⚠️ Son los números **de su paper (2024)**: el campo se movió (ViDoRe V2 y V3, ColQwen, Jina v4…).

### 6.1 MaxSim sobre patches
Con la query tokenizada en vectores $q_1..q_m$ y la página en patches $p_1..p_n$ (ambos en $\mathbb R^D$, normalizados):
$$\mathrm{score}(q,\text{página})=\sum_{i=1}^{m}\max_{1\le j\le n}\ \langle q_i,\,p_j\rangle$$
Cada palabra de la query **elige el patch que mejor la «ve»**; sumamos esas coincidencias. Un vector medio de la página tendría que contener *todo* a la vez.

> ### 🎬 GEMINI-VIDEO · ID: `M16-VID-1` · ⚠️ REEMPLAZAR ESTA CELDA
> **ColPali: de la página a los patches y MaxSim**
>
> **Qué muestra:** Animación del recorrido: página → rejilla de patches → un vector por patch; query → un vector por token; cada token toma el máximo sobre los patches; se suman los máximos.
> **Formato:** MP4 16:9, 1080p | Duración: 30 s | **Texto alternativo:** Una página se divide en patches con un vector cada uno; cada token de la consulta toma su máxima similitud sobre los patches y se suman.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Animación técnica limpia, 16:9, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif, ritmo calmado, sin voz en off, sin música. Textos en español, términos técnicos en inglés.
> Escenas:
> 0–5 s: aparece una página de documento esquemática (rectángulo blanco con borde gris; un título como barra gris, dos párrafos como líneas grises, un gráfico de barras azules y una pequeña tabla). Título en pantalla: «Una página = una imagen».
> 5–11 s: sobre la página se dibuja una rejilla de 4×3 celdas y cada celda se resalta brevemente de izquierda a derecha, de arriba abajo; a la derecha aparece una columna de 12 vectores pequeños (barras verticales de colores) con la etiqueta «1 vector por patch (128-d)».
> 11–16 s: abajo a la izquierda aparece la consulta «latencia del índice HNSW»; se parte en tres tokens, «latencia», «índice», «HNSW», y cada uno genera un vector naranja con la etiqueta «1 vector por token».
> 16–25 s: para el token «índice» se dibujan 12 líneas finas grises hacia los 12 vectores de patch y la de mayor similitud se engrosa en verde con el texto «max»; se repite para «latencia» y «HNSW», cada uno con su línea verde hacia un patch distinto (el gráfico de barras, la tabla y el párrafo).
> 25–30 s: los tres valores máximos se suman en una caja verde: «score = max₁ + max₂ + max₃» y aparece el texto «MaxSim (late interaction)».
> Evita: voz en off, música, texto ilegible, números inventados, efectos 3D, transiciones bruscas, imágenes realistas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M16-VID-1.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M16-VID-1.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> página -> [p1 p2 ... p12]   (un vector por patch)
> query  -> [q1 q2 q3]        (un vector por token)
> score = max_j<q1,pj> + max_j<q2,pj> + max_j<q3,pj>
> ```

> 👀 **Qué debes observar.** Cada token de la query «apunta» a un **patch distinto** (el gráfico, la tabla, el párrafo): por eso MaxSim distingue páginas que contienen *todos* los conceptos de las que solo tienen la media de ellos. El mapa de calor que construiremos en 6.3 es esta misma operación.

### 6.2 ColPali en miniatura (entrenable en CPU)
Construimos páginas de 12 celdas (3×4); cada celda contiene **una palabra renderizada** (fuente real, con desplazamiento aleatorio), un **icono** (forma de color: contenido *no textual*) o está vacía. Entrenamos un codificador de patch (CNN) contra una tabla de embeddings de «tokens» (palabras e iconos) con ruido de píxel. Es el equivalente funcional del VLM de ColPali: *patch de imagen → vector en el mismo espacio que los tokens de la query*.

In [ ]:
import math, unicodedata
from PIL import Image, ImageDraw, ImageFont

try: FUENTE = ImageFont.load_default(size=11)      # Pillow >= 10.1 trae una fuente escalable
except TypeError: FUENTE = ImageFont.load_default()
CH, CW, FILAS, COLS = 16, 96, 3, 4                  # celda de 16x96 px; página = 3x4 celdas = 12 "patches"

def plano(s: str) -> str:
    """La fuente por defecto no tiene tildes: las plegamos (la limitación es de la FUENTE, no del enfoque)."""
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def celda_palabra(w: str, rng=None) -> np.ndarray:
    img = Image.new("L", (CW, CH), 255); dx = int(rng.integers(0, 6)) if rng is not None else 0; dy = int(rng.integers(0, 3)) if rng is not None else 0
    ImageDraw.Draw(img).text((2 + dx, 1 + dy), w, font=FUENTE, fill=0); return np.asarray(img, dtype=np.float32) / 255.

def celda_icono(k: int, rng) -> np.ndarray:
    """Icono (forma de color) 16x16 sobre una celda blanca: contenido NO textual (como un logo o una figura de la página)."""
    f, c = CONCEPTOS[k]; ic = dibuja(f, c, rng, S=16); lum = {"red": .35, "green": .6, "blue": .1, "yellow": .85}[c]   # el color -> nivel de gris
    mask = np.abs(ic - .5).max(-1) > .25; cel = np.ones((CH, CW), np.float32); x0 = int(rng.integers(0, CW - 16))
    cel[:, x0:x0 + 16] = np.where(mask, lum, 1.0); return cel

def celda_vacia() -> np.ndarray: return np.ones((CH, CW), np.float32)

def pagina_a_imagen(celdas, ruido: float, rng) -> np.ndarray:
    """Une 12 celdas en una imagen de 48x384 px y añade ruido gaussiano de píxel (escaneo/compresión)."""
    a = np.stack(celdas).reshape(FILAS, COLS, CH, CW).transpose(0, 2, 1, 3).reshape(FILAS * CH, COLS * CW)
    return a + rng.normal(0, ruido, a.shape).astype(np.float32)

def parchea(pagina: np.ndarray) -> np.ndarray:
    """Imagen de página -> (12, 16, 96): los 'patches' (celdas). Un ViT real usa cuadrados de 14-16 px; aquí cada patch es una celda."""
    return pagina.reshape(FILAS, CH, COLS, CW).transpose(0, 2, 1, 3).reshape(FILAS * COLS, CH, CW)

class RedPatch(nn.Module):
    """Codificador de patch: CNN -> vector L2-normalizado de dimensión D (como la proyección a 128-d de ColPali, en pequeño)."""
    def __init__(self, D: int = 32):
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((2, 6)), nn.Flatten(), nn.Linear(64 * 12, D))
    def forward(self, x): return F.normalize(self.c(x.unsqueeze(1)), dim=-1)

def maxsim(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    """Late interaction (ColBERT/ColPali): score(q, pág) = sum_i max_j <q_i, p_j>.
    Q: (nq, D) tokens de la query; P: (n_págs, n_patches, D). Devuelve (n_págs,). Vectorizado con einsum."""
    return np.einsum("qd,npd->nqp", Q, P).max(axis=2).sum(axis=1)

def ocr_plantillas(palabras, rng_seed=0):
    """OCR 'clásico' por correlación: plantillas de cada palabra en TODOS los desplazamientos (dx, dy). Es un filtro adaptado,
    casi óptimo contra ruido gaussiano para texto renderizado con esta fuente. Devuelve (T, etiqueta de cada plantilla)."""
    T, lab = [celda_vacia().ravel()], [0]
    for k, w in enumerate(palabras, 1):
        for dx in range(6):
            for dy in range(3):
                img = Image.new("L", (CW, CH), 255); ImageDraw.Draw(img).text((2 + dx, 1 + dy), w, font=FUENTE, fill=0)
                T.append(np.asarray(img, dtype=np.float32).ravel() / 255.); lab.append(k)
    return np.stack(T), np.array(lab)

def ocr_pagina(parches: np.ndarray, T: np.ndarray, lab: np.ndarray) -> np.ndarray:
    """Reconoce cada patch como la palabra de la plantilla más cercana (L2). Los iconos se 'leen' mal: no son texto."""
    d = torch.cdist(torch.tensor(parches.reshape(len(parches), -1)), torch.tensor(T)); return lab[d.argmin(1).numpy()]

In [ ]:
PALABRAS = ("cache index token query model vector graph latency memory thread lock tests branch merge cloud shard route kernel buffer socket "
            "digest cursor schema trace metric batch queue stack hash layer loss epoch gamma delta cosine matrix tensor decode patch retry pool").split()[:40]
NW = len(PALABRAS); V = 1 + NW + 16            # clase 0 = celda vacía · 1..NW palabras · NW+1.. iconos (los 16 conceptos forma×color)
nombre_clase = ["<vacía>"] + PALABRAS + [f"icono:{c}-{f}" for f, c in CONCEPTOS]

def muestra_celdas(n, rng, ruido_max=.3):
    ys = rng.integers(0, V, n); xs = []
    for y in ys:
        c = celda_vacia() if y == 0 else celda_palabra(PALABRAS[y - 1], rng) if y <= NW else celda_icono(y - NW - 1, rng)
        xs.append(c + rng.normal(0, float(rng.uniform(0, ruido_max)), c.shape).astype(np.float32))
    return torch.tensor(np.stack(xs)), torch.tensor(ys)

def entrena_red_patch(pasos=700, D=32, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); red = RedPatch(D)
    emb = nn.Parameter(torch.randn(V, D) * .1); logt = nn.Parameter(torch.tensor(math.log(10.)))
    opt = torch.optim.Adam(list(red.parameters()) + [emb, logt], lr=3e-3); hist = []
    for _ in range(pasos):
        x, y = muestra_celdas(128, rng); loss = F.cross_entropy(logt.exp() * red(x) @ F.normalize(emb, dim=-1).T, y)
        opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
    return red.eval(), F.normalize(emb, dim=-1).detach().numpy(), hist

t0 = time.time(); RED, EMB, hist_patch = entrena_red_patch(700 if FAST_DEV_RUN else 2000)
print(f"codificador de patch entrenado en {time.time()-t0:.0f}s · pérdida final {np.mean(hist_patch[-20:]):.3f}")

In [ ]:
rng_p = np.random.default_rng(11); N_PAG = 200 if FAST_DEV_RUN else 600
def genera_paginas(n):
    G = []
    for _ in range(n):
        nw, ni = int(rng_p.integers(4, 7)), int(rng_p.integers(1, 3))
        cont = np.concatenate([rng_p.choice(NW, nw, replace=False) + 1, rng_p.choice(16, ni, replace=False) + NW + 1])
        g = np.zeros(12, int); g[rng_p.choice(12, len(cont), replace=False)] = cont; G.append(g)
    return np.stack(G)
GRID = genera_paginas(N_PAG)
def render(g, rng, ruido):
    return pagina_a_imagen([celda_vacia() if c == 0 else celda_palabra(PALABRAS[c - 1], rng) if c <= NW else celda_icono(c - NW - 1, rng) for c in g], ruido, rng)
fig, ax = plt.subplots(1, 3, figsize=(13, 2.6))
for a, ruido in zip(ax, (0.0, 0.3, 0.45)):
    a.imshow(render(GRID[0], np.random.default_rng(0), ruido), cmap="gray", vmin=0, vmax=1); a.set_title(f"página 0 · ruido de píxel σ={ruido}", fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()
print("contenido de la página 0:", [nombre_clase[c] for c in GRID[0] if c])

> 👀 **Qué debes observar.** Con σ=0.45 el texto sigue siendo legible para una persona pero ya es difícil para un modelo pequeño. Los **iconos** no tienen ninguna palabra: un OCR no puede «leerlos»; un codificador visual sí puede alinearlos con tokens de la query (p. ej. «icono:red-circle»).

In [ ]:
TPL, LAB = ocr_plantillas(PALABRAS)            # OCR clásico por plantillas con desplazamientos
rq = np.random.default_rng(13)
def ndcg10(order, rel):
    rs = set(rel); dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(order[:10]) if d in rs); idcg = sum(1 / math.log2(i + 2) for i in range(min(10, len(rs)))); return dcg / idcg
def consultas(tipo, n):
    out = []
    for _ in range(n):
        g = GRID[rq.integers(0, N_PAG)]; w, ic = g[(g > 0) & (g <= NW)], g[g > NW]
        out.append(rq.choice(w, 2, replace=False) if tipo == "2 palabras" else np.array([rq.choice(w), rq.choice(ic)]) if tipo == "1 palabra + 1 icono" else (rq.choice(ic, 2, replace=False) if len(ic) > 1 else np.array([ic[0]])))
    return out
TIPOS = ["2 palabras", "1 palabra + 1 icono", "2 iconos"]; N_Q = 80 if FAST_DEV_RUN else 200
def evalua(ruido):
    rr = np.random.default_rng(20); P = np.stack([parchea(render(g, rr, ruido)) for g in GRID])
    with torch.no_grad(): PE = RED(torch.tensor(P.reshape(-1, CH, CW))).reshape(N_PAG, 12, -1).numpy()
    ocr = np.stack([ocr_pagina(P[i], TPL, LAB) for i in range(N_PAG)]); sets = [set(o) for o in ocr]; res = {}
    for tipo in TIPOS:
        r = {"MaxSim (multi-vector)": [], "vector medio (1 por página)": [], "OCR + coincidencia de palabras": []}
        for q in consultas(tipo, N_Q):
            rel = [j for j in range(N_PAG) if all(w in GRID[j] for w in q)]; qe = EMB[q]
            for k, s in (("MaxSim (multi-vector)", maxsim(qe, PE)), ("vector medio (1 por página)", PE.mean(1) @ qe.mean(0)), ("OCR + coincidencia de palabras", np.array([sum(float(w in st) for w in q) for st in sets]))):
                r[k].append(ndcg10(np.lexsort((np.arange(N_PAG), -s)), rel))
        res[tipo] = {k: float(np.mean(v)) for k, v in r.items()}
    return res, PE
t0 = time.time(); RUIDOS = (0.0, 0.3, 0.45); RES_PAG = {}
for rz in RUIDOS: RES_PAG[rz] = evalua(rz)[0]
print(f"({time.time()-t0:.0f}s)"); pd.DataFrame({f"σ={rz} · {t}": v for rz in RUIDOS for t, v in RES_PAG[rz].items()}).T.round(2)

In [ ]:
sist = ["MaxSim (multi-vector)", "vector medio (1 por página)", "OCR + coincidencia de palabras"]; cols = [AZUL, GRIS, NARANJA]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for a, rz in zip(ax, RUIDOS):
    for i, (s, c) in enumerate(zip(sist, cols)): a.bar(np.arange(3) + (i - 1) * .27, [RES_PAG[rz][t][s] for t in TIPOS], .27, color=c, label=s)
    a.set_xticks(range(3)); a.set_xticklabels(TIPOS, fontsize=8); a.set_title(f"ruido de píxel σ = {rz}")
ax[0].set_ylabel("nDCG@10"); ax[0].legend(frameon=False, fontsize=7); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) El **vector medio** falla siempre: promediar patches *diluye* cada palabra entre las demás (el cuello de botella de un único vector por página). (2) Con texto limpio, **OCR + palabras es perfecto** (nDCG≈1) en consultas de palabras: no hay razón para pagar un modelo visual si el texto es legible y tu pregunta es léxica. (3) En consultas que incluyen **iconos**, el OCR cae porque no hay texto que leer, y MaxSim sigue funcionando (imperfecto: el codificador es diminuto). (4) Con mucho ruido, **el OCR «filtro adaptado» aguanta mejor que el codificador pequeño** en palabras: un OCR clásico bien afinado es un baseline duro.
> Conclusión honesta: el embedding visual **no sustituye** al OCR en texto limpio; gana donde el contenido relevante **no es texto** o donde OCR/parseo es frágil.

In [ ]:
# Mapa de calor token-de-query × patch: la "explicabilidad" de ColPali. Usamos una consulta con dos palabras de la página 0.
rr = np.random.default_rng(0); pag = render(GRID[0], rr, 0.1); P0 = parchea(pag)
with torch.no_grad(): pe = RED(torch.tensor(P0)).numpy()
ws = [c for c in GRID[0] if 0 < c <= NW][:2]; qe = EMB[ws]; sim = qe @ pe.T          # (2, 12)
fig, ax = plt.subplots(1, 3, figsize=(14, 2.8), gridspec_kw={"width_ratios": [2.2, 1, 1]}); ax[0].imshow(pag, cmap="gray", vmin=0, vmax=1); ax[0].set_title("página (σ=0.1)", fontsize=9); ax[0].axis("off")
for a, w, s in zip(ax[1:], ws, sim):
    a.imshow(s.reshape(FILAS, COLS), cmap="viridis", vmin=-.2, vmax=1); a.set_title(f"similitud del token «{PALABRAS[w-1]}»", fontsize=9); a.set_xticks([]); a.set_yticks([])
    j = int(s.argmax()); a.add_patch(plt.Rectangle((j % COLS - .5, j // COLS - .5), 1, 1, fill=False, ec=ROJO, lw=2))
plt.tight_layout(); plt.show()
print("score MaxSim de la página 0 para esa query:", round(float(maxsim(qe, pe[None])[0]), 3), "· máximo posible =", len(ws))

> 👀 **Qué debes observar.** Cada token ilumina **exactamente la celda** donde está su palabra (recuadro rojo) y el resto queda cerca de 0. Esos mapas son los que usan ColPali/ViDoRe para mostrar *qué región de la página* justificó el resultado: mejor depuración que un score opaco.

### 6.3 Coste: ¿cuánto pesa un índice multi-vector?
ColPali guarda ≈ 1 030 vectores de 128 dimensiones por página: en `float16`, $1030\times128\times2\ \text{B}=263\,680\ \text{B}\approx257.5\ \text{KiB}$ (el paper reporta ≈ 257.5 KB por página). Un único vector de 1024-d en `float16` son 2 KiB. **Calcula tú el coste para tu colección.**

In [ ]:
def bytes_pagina(n_vec, dim, bytes_por_valor): return n_vec * dim * bytes_por_valor
esquemas = {"ColPali (1030×128, fp16)": bytes_pagina(1030, 128, 2), "ColPali binarizado (1030×128 bits)": 1030 * 128 / 8, "token pooling ÷4 (≈258×128, fp16)": bytes_pagina(258, 128, 2),
            "1 vector 1024-d fp16 (DSE / denso)": bytes_pagina(1, 1024, 2), "1 vector 1024-d binario": 1024 / 8, "texto OCR + embedding 1024-d fp16 (≈ 3 KB de texto)": 3000 + bytes_pagina(1, 1024, 2)}
paginas = [10**4, 10**6, 10**8]
df = pd.DataFrame({n: [b * p / 1e9 for p in paginas] for n, b in esquemas.items()}, index=[f"{p:,} páginas" for p in paginas]).T.round(2); display(df.rename(columns=lambda c: c + " (GB)"))
fig, ax = plt.subplots(figsize=(8.5, 3.6)); ax.barh(list(esquemas), [b / 1024 for b in esquemas.values()], color=[AZUL, AZUL, AZUL, GRIS, GRIS, NARANJA]); ax.set_xscale("log")
ax.set_xlabel("KiB por página (escala log)"); ax.set_title("El multi-vector cuesta ~100× más almacenamiento por página"); ax.invert_yaxis(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Para 1 M de páginas, ColPali en `fp16` ocupa **≈ 264 GB** frente a ≈ 2 GB de un vector denso: el multi-vector se paga en RAM/SSD y en cómputo de MaxSim. Las palancas son: **binarizar**, *token pooling* (agrupar patches), reducir $D$, **pre-filtrar** con un vector por página y reordenar con MaxSim, o **codificaciones de dimensión fija** como MUVERA (Dhulipala et al., 2024: convierten multi-vector en un vector único para usar ANN). 🟡 Los factores de las filas «binarizado» y «pooling» son aritmética sobre el esquema; la **pérdida de calidad** hay que medirla (módulo 14 y 15).

### 6.4 Con modelos reales: ColPali / ColQwen y ViDoRe **(opcional, no probada en CPU)**
En Colab con GPU (L4 recomendada, bf16) puedes usar el paquete `colpali-engine`. La API siguiente es la del README del paquete (PyPI): consulta la versión instalada porque los nombres de checkpoints y métodos evolucionan. **Esta celda no se pudo ejecutar en el entorno de desarrollo (sin acceso a Hugging Face ni GPU).**

In [ ]:
RUN_COLPALI = False      # True en Colab con GPU. pip install -q colpali-engine
if RUN_COLPALI:
    from colpali_engine.models import ColPali, ColPaliProcessor
    from PIL import Image
    nombre = "vidore/colpali-v1.3"                       # alternativas más recientes: familia ColQwen2/2.5 (mismo patrón de API)
    modelo = ColPali.from_pretrained(nombre, torch_dtype=torch.bfloat16, device_map=device).eval(); proc = ColPaliProcessor.from_pretrained(nombre)
    paginas_img = [Image.fromarray((render(GRID[i], np.random.default_rng(i), 0.0) * 255).astype("uint8")).convert("RGB").resize((448, 448)) for i in range(4)]
    consultas_txt = ["cache index", "token query"]
    with torch.no_grad():
        emb_pag = modelo(**proc.process_images(paginas_img).to(modelo.device)); emb_q = modelo(**proc.process_queries(consultas_txt).to(modelo.device))
    print(proc.score_multi_vector(emb_q, emb_pag))       # (n_queries, n_páginas)
else:
    print("RUN_COLPALI=False: celda omitida. En Hugging Face Transformers también existe ColPaliForRetrieval (checkpoint «vidore/colpali-v1.3-hf»).")

**Benchmarks.** *ViDoRe* (Faysse et al., 2024) evalúa retrieval a nivel de página con nDCG@5; **ViDoRe V2** (Macé et al., 2025) sube el listón; **ViDoRe V3** (arXiv ene-2026, ACL 2026) cubre 10 datasets profesionales, ≈ 26 000 páginas y 3 099 queries en 6 idiomas, con anotación humana; concluye que los retrievers visuales superan a los de solo texto y que late interaction y reranking de texto ayudan, pero que **siguen fallando con elementos no textuales, preguntas abiertas y grounding fino** (a oct-2026; fuente: su paper). MIEB (Xiao et al., 2025) es el benchmark masivo de embeddings de imagen.

## 7. Embeddings unificados multimodales
La tendencia 2025–2026 es **un solo modelo y un solo espacio** para texto, imágenes, vídeo, audio y documentos, en lugar de una torre por modalidad.

| Modelo | Qué ofrece (fuente) | Madurez |
|---|---|---|
| **Gemini Embedding 2** (Google) | Primer modelo de embeddings nativamente multimodal de Google: texto, imágenes, vídeo, audio y PDFs en un único espacio; >100 idiomas; Matryoshka (dim. por defecto 3 072). Lanzado en *preview* el **10-mar-2026** y GA el **22-abr-2026**. Según la cobertura de prensa que pude consultar (no pude abrir blog.google/ai.google.dev desde mi entorno), límites por petición de ≈ 8 192 tokens de texto, 6 imágenes, vídeo hasta 120 s, audio hasta 80 s y PDFs de hasta 6 páginas, y **espacio incompatible con `gemini-embedding-001`** (hay que re-embeber). **Verifica límites y precios en la documentación oficial.** | 🟡 |
| **jina-embeddings-v4** (Jina AI) | Texto + imágenes (incluidos documentos visuales); produce **vectores densos y multi-vector (late interaction)** (Günther et al., 2025) | 🟡 |
| **VLM2Vec / MMEB** | Convierte VLMs en embedders multimodales con un benchmark de 36 tareas (Jiang et al., 2024) | 🟡 / 🔴 |
| **ImageBind** (Meta) | Un espacio de 6 modalidades con *imágenes como pivote* (Girdhar et al., CVPR 2023) | 🟡 |
| **CLAP** | Audio ↔ texto con contraste (Elizalde et al., 2022) | 🟡 |

> 📅 Las capacidades de modelos propietarios cambian: la tabla anterior es **a oct-2026** y sujeta a la fecha de cada fuente. El paper de *Gemini Embedding* (Lee et al., 2025, arXiv 2503.07891) describe el modelo de **texto**; la versión multimodal «2» se documenta en anuncios y documentación de Google.

**(opcional, protegida)** Llamada a la API de Gemini para embeber un texto y una imagen y medir su similitud. Solo se ejecuta si defines `GEMINI_API_KEY`; el id del modelo lo lees de `GEMINI_EMBED_MODEL` (por defecto `gemini-embedding-2`; en *preview* se llamó `gemini-embedding-2-preview`). Sigue el patrón del SDK `google-genai`; **verifica nombres de parámetros en la documentación vigente**.

In [ ]:
if os.environ.get("GEMINI_API_KEY"):
    from google import genai
    from google.genai import types
    import io
    from PIL import Image
    cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"]); modelo_id = os.environ.get("GEMINI_EMBED_MODEL", "gemini-embedding-2")
    buf = io.BytesIO(); Image.fromarray((Xte[0] * 255).astype("uint8")).resize((224, 224)).save(buf, format="PNG")
    r_txt = cliente.models.embed_content(model=modelo_id, contents=[" ".join(Tte[0])], config=types.EmbedContentConfig(output_dimensionality=768))
    r_img = cliente.models.embed_content(model=modelo_id, contents=[types.Part.from_bytes(data=buf.getvalue(), mime_type="image/png")], config=types.EmbedContentConfig(output_dimensionality=768))
    a, b = (np.array(r.embeddings[0].values) for r in (r_txt, r_img)); a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)   # normalizar al truncar (Matryoshka)
    print("coseno texto↔imagen:", float(a @ b))
else:
    print("Sin GEMINI_API_KEY: celda omitida (opcional).")

## 8. Audio y vídeo en espacios de embedding
- **Audio**: CLAP (audio ↔ texto) e ImageBind; para voz, la ruta habitual sigue siendo **ASR (p. ej. Whisper) → texto → embedder**, que conserva la exactitud léxica. Los embeddings de audio nativos (Gemini Embedding 2 declara audio sin transcripción) ganan cuando importa el sonido no verbal (música, eventos) o se quiere evitar el coste/latencia de ASR. 🟡
- **Vídeo**: se suele **muestrear fotogramas** (p. ej. 1–2 fps), embeberlos y combinarlos. Lo que se pierde al combinar es el **orden temporal**. Lo medimos.

In [ ]:
def video(forma, color, direccion, rng, n_frames=8):
    """Una figura que cruza el cuadro de izquierda a derecha (direccion=+1) o al revés (-1)."""
    xs = np.linspace(8, 24, n_frames)[::direccion]; cy = int(rng.integers(14, 19))
    return np.stack([dibuja(forma, color, rng, centro=(int(x), cy)) for x in xs])
rng_v = np.random.default_rng(5); I_ = MODELOS["InfoNCE τ aprendida"]["I"]; Vs, Dir, Col = [], [], []
for _ in range(160):
    d = int(rng_v.choice([1, -1])); k = int(rng_v.integers(0, 16)); f, c = CONCEPTOS[k]
    Vs.append(video(f, c, d, rng_v)); Dir.append(d); Col.append(k)
with torch.no_grad(): FE = np.stack([I_(torch.tensor(v)).numpy() for v in Vs])      # (160, 8, D)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
medio = FE.mean(1); secuencia = FE.reshape(len(FE), -1); Dir = np.array(Dir)
acc = {"vector medio de los frames": cross_val_score(LogisticRegression(max_iter=2000), medio, Dir, cv=5).mean(),
       "frames concatenados (conserva el orden)": cross_val_score(LogisticRegression(max_iter=2000), secuencia, Dir, cv=5).mean()}
acc_col = cross_val_score(LogisticRegression(max_iter=2000), medio, np.array(Col), cv=5).mean()
ax = pd.Series(acc).plot(kind="bar", color=[ROJO, VERDE], figsize=(6, 3.2), rot=10); ax.axhline(.5, color=GRIS, ls="--"); ax.set_ylim(0, 1.05); ax.set_ylabel("precisión al predecir la dirección")
ax.set_title("Promediar frames borra el orden temporal"); plt.tight_layout(); plt.show(); print(f"(el mismo vector medio SÍ identifica el concepto forma×color: precisión {acc_col:.2f})")

> 👀 **Qué debes observar.** El vector medio de frames identifica **qué** aparece (forma y color) pero **no hacia dónde se mueve**: la dirección queda al azar (≈ 0.5) porque la media es invariante al orden. Conservar el orden (concatenar, atención temporal, modelos de vídeo nativos) lo resuelve en el juguete.
> Implicación práctica: un embedding de vídeo basado en frames sirve para «encontrar el clip donde sale X», no para «el clip donde alguien *entra* en vez de *salir*».

## 9. ¿OCR + texto o embedding visual? Árbol de decisión

> ### 🎨 GEMINI-IMAGEN · ID: `M16-IMG-3` · ⚠️ REEMPLAZAR ESTA CELDA
> **Árbol de decisión: OCR + texto vs embedding visual**
>
> **Qué muestra:** Guía de decisión para elegir el pipeline de documentos de Políglota según el tipo de contenido, la calidad del OCR y las restricciones de coste.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Árbol de decisión con tres preguntas que lleva a parseo+texto, OCR de calidad, ColPali multi-vector o un vector por página.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama de flujo técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Árbol de decisión de arriba abajo con rombos (preguntas) y rectángulos redondeados (resultados). Textos literales:
> Rombo 1: «¿El contenido relevante es texto limpio y seleccionable?» → Sí: rectángulo verde «Parseo + texto + embedder de texto (+ BM25 híbrido)». → No: siguiente.
> Rombo 2: «¿Hay tablas, gráficos, capturas o diagramas que responden preguntas?» → No: rectángulo azul «OCR de calidad + texto». → Sí: siguiente.
> Rombo 3: «¿Presupuesto de almacenamiento multi-vector (≈ 250 KB/página)?» → Sí: rectángulo naranja «ColPali / ColQwen (multi-vector) + MaxSim». → No: rectángulo naranja «Un vector por página (DSE) o pooling/binarización + reranking».
> A la derecha, un recuadro gris: «Siempre: evaluar con IC por tipo de consulta (módulo 15). Considerar híbrido: texto + visual con fusión RRF».
> Título superior: «¿OCR + texto o embedding visual?».
> Evita: texto ilegible o inventado, ilustraciones decorativas, más de tres rombos, flechas que se crucen.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M16-IMG-3.png` en la carpeta del módulo y reemplaza esta celda por: `![Árbol de decisión con tres preguntas que lleva a parseo+texto, OCR de calidad, ColPali multi-vector o un vector por página.](assets/M16-IMG-3.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> texto limpio? --sí--> parseo + texto + embedder (+BM25)
>    no -> ¿hay tablas/figuras que responden? --no--> OCR de calidad + texto
>             sí -> ¿presupuesto multi-vector? --sí--> ColPali/ColQwen + MaxSim ; no -> 1 vector/página o pooling + rerank
> ```

> 👀 **Qué debes observar.** La primera pregunta evita pagar por visión cuando no hace falta (nuestro experimento: OCR perfecto en texto limpio). La tercera es de **coste**: el multi-vector no es gratis (§6.3). La mejor práctica actual es **híbrida**: índice de texto + índice visual y fusión por rangos.
> Idea para Políglota: la documentación técnica tiene muchos diagramas de arquitectura y capturas de consola: ahí el índice visual aporta; en README y referencias API, no.

## 10. 🏭 En producción
- **Un índice por modalidad, fusión por rangos.** Mezclar vectores de texto e imagen en un solo ANN sin calibrar sufre el *modality gap* (§5). Lo habitual es consultar índices separados y fusionar con RRF (módulo 13), o usar un modelo unificado entrenado para ello.
- **Documentos visuales:** los proveedores de embeddings (Google con Gemini Embedding 2, Jina con v4, Cohere, Voyage…) ofrecen API multimodal; los equipos que quieren control usan ColPali/ColQwen con *late interaction* y aceleran con pre-filtro + reordenado. Los costes dominantes son **almacenamiento** y **latencia de MaxSim**, no la calidad.
- **Re-embedding:** cambiar de modelo multimodal exige re-procesar **imágenes/PDFs** (más caro que texto). Guarda las páginas renderizadas y versiona el modelo (módulo 18).
- **Privacidad:** capturas de pantalla contienen datos sensibles (tokens, correos): decide qué se envía a APIs externas.
- **Evaluación:** construye un golden set de *queries visuales* (preguntas sobre tablas/figuras) con anotación humana y mide por tipo de consulta con IC (módulo 15).

## 11. 🧠 Secretos de la élite
1. **El gap casi nunca importa para ranking dentro de una modalidad, y siempre importa al mezclar.** Medido aquí: con τ=0.01 (gap ≈ 1) el 100 % de las queries tiene un texto equivocado por delante de la imagen correcta en un índice mixto, y restar el centroide por modalidad **no** lo arregla (§5); con gap ≤ 0.35 no hubo intrusos.
2. **La temperatura es un hiperparámetro de geometría, no solo de optimización**: $\tau$ baja ⇒ gap alto (Liang et al.; nuestro barrido). Al comparar dos modelos, compara también su $\tau$ aprendida.
3. **Los falsos negativos en el batch** (varias imágenes del mismo concepto) ponen un suelo a la pérdida contrastiva: con pocos conceptos y batches grandes, la pérdida no baja de $\ln(N/\text{conceptos})$ aunque el modelo sea perfecto.
4. **El coste real del multi-vector es el almacenamiento**: ≈ 257.5 KB/página en ColPali (fp16) ⇒ ≈ 257 GB por millón de páginas; decide pooling/binarización/pre-filtro *antes* de indexar.
5. **Un OCR clásico bien afinado es un baseline duro** en texto limpio (§6.2). Compara siempre contra «OCR + BM25 + embedder de texto» por tipo de consulta antes de adoptar visión.
6. **Un vector medio de frames/patches es invariante al orden y diluye detalles**: sirve para «qué hay», no para «dónde/cuándo/en qué orden».
7. **Verifica límites por petición y compatibilidad de espacio al migrar a un embedder multimodal propietario**: p. ej., la prensa reporta que Gemini Embedding 2 no comparte espacio con `gemini-embedding-001` (hay que re-embeber).
8. **Benchmarks de documentos (ViDoRe V3) muestran que los retrievers visuales ganan a los de texto, pero siguen flojos con elementos no textuales y grounding fino** (a oct-2026): no vendas «OCR-free resuelve todo».

## 12. ⚠️ Errores comunes
- Comparar similitudes **entre modalidades** (texto↔texto vs texto↔imagen) como si estuvieran en la misma escala.
- Olvidar **normalizar** al truncar dimensiones (Matryoshka) en APIs multimodales.
- Evaluar retrieval visual solo con queries *léxicas* (que un OCR resuelve) y concluir que «la visión no aporta» — o con queries visuales y concluir que «OCR no sirve».
- Usar fuentes/renderizados de entrenamiento distintos a producción (resolución, DPI, idioma) sin medir.
- Indexar páginas a baja resolución para ahorrar y perder texto pequeño.
- Tratar el espacio de un modelo propietario como estable: **versiona** modelo y dimensión.
- Promediar frames y esperar sensibilidad temporal.

## 13. ✅ Autoevaluación
<details><summary><b>1. (Cálculo)</b> Una colección de 2 millones de páginas con ColPali (1 030 vectores × 128 d, fp16). ¿Cuánto ocupa? ¿Y binarizada a 1 bit por dimensión?</summary>

$1030\times128\times2=263\,680$ B/página ⇒ $2\times10^6\times263\,680\approx527$ GB. Binarizada: $1030\times128/8=16\,480$ B ⇒ ≈ 33 GB (16× menos), a costa de calidad que debes medir.
</details>

<details><summary><b>2. (Cálculo)</b> Query de 3 tokens, página de 4 patches; similitudes (filas = tokens): [[.9,.1,.0,.2],[.1,.8,.3,.0],[.0,.2,.7,.1]]. ¿Score MaxSim? ¿Y con el vector medio si los vectores fueran ortonormales?</summary>

Máximos por fila: .9 + .8 + .7 = **2.4**. Con el vector medio (promedio de patches y de tokens) cada token solo contribuye con ≈ 1/4 de su coincidencia: la señal se diluye (por eso el vector medio falló en nuestro experimento).
</details>

<details><summary><b>3. (Diagnóstico)</b> Tu índice mixto (imágenes + textos) devuelve casi solo textos para una query de texto. ¿Causa y arreglos?</summary>

Modality gap: los textos están más cerca entre sí que de las imágenes. Arreglos: índices separados + RRF; un modelo entrenado para un espacio unificado; centrar cada modalidad (restar su centroide y renormalizar) ayuda a veces, pero en nuestro experimento extremo no bastó. Mide con un golden set mixto.
</details>

<details><summary><b>4. (Diagnóstico)</b> ColPali brilla en ViDoRe pero en tus PDFs financieros (tablas pequeñas, texto denso) un pipeline OCR+texto le gana. Hipótesis.</summary>

Resolución de entrada insuficiente para texto pequeño; dominio distinto al de entrenamiento; las queries son léxicas (identificadores, cifras exactas) donde BM25/OCR son mejores; ViDoRe no representa tu distribución. Evalúa por tipo de consulta e híbrido.
</details>

<details><summary><b>5. (Transferencia)</b> Quieres buscar clips de vídeo de demos de producto en Políglota. ¿Qué pipeline propones y qué no cubrirá?</summary>

Muestreo de frames (+ ASR de la voz → texto) y embedder multimodal; fusión de índices de texto (transcripción) y visual. No cubrirá bien el **orden/dirección temporal** ni acciones finas: añade metadatos o un modelo de vídeo nativo.
</details>

<details><summary><b>6. (Transferencia)</b> ¿Cómo repetirías el experimento de modality gap con CLIP real y qué medirías?</summary>

Embeder N pares imagen–caption de COCO con `get_image_features/get_text_features`, normalizar, medir ‖centroide_img − centroide_txt‖, el coseno medio intra-modalidad y la «tasa de intrusos» en un índice mixto antes/después de centrar. Comparar CLIP vs SigLIP/SigLIP 2.
</details>

## 14. 📚 Referencias
- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision* (CLIP). <https://arxiv.org/abs/2103.00020>
- Zhai, Mustafa, Kolesnikov, Beyer (2023). *Sigmoid Loss for Language Image Pre-Training* (SigLIP). <https://arxiv.org/abs/2303.15343>
- Tschannen et al. (2025). *SigLIP 2: Multilingual Vision-Language Encoders with Improved Semantic Understanding, Localization, and Dense Features*. <https://arxiv.org/abs/2502.14786>
- Liang, Zhang, Kwon, Yeung, Zou (2022). *Mind the Gap: Understanding the Modality Gap in Multi-modal Contrastive Representation Learning*. NeurIPS. <https://arxiv.org/abs/2203.02053>
- Faysse et al. (2024). *ColPali: Efficient Document Retrieval with Vision Language Models*. ICLR 2025. <https://arxiv.org/abs/2407.01449>
- Ma, Lin, Li, Chen, Lin (2024). *Unifying Multimodal Retrieval via Document Screenshot Embedding*. EMNLP. <https://arxiv.org/abs/2406.11251>
- Macé, Loison, Faysse (2025). *ViDoRe Benchmark V2: Raising the Bar for Visual Retrieval*. <https://arxiv.org/abs/2505.17166>
- ViDoRe V3 (2026). *ViDoRe V3: A Comprehensive Evaluation of Retrieval Augmented Generation in Complex Real-World Scenarios*. <https://arxiv.org/abs/2601.08620>
- Günther et al. (2025). *jina-embeddings-v4: Universal Embeddings for Multimodal Multilingual Retrieval*. <https://arxiv.org/abs/2506.18902>
- Jiang et al. (2024). *VLM2Vec: Training Vision-Language Models for Massive Multimodal Embedding Tasks*. <https://arxiv.org/abs/2410.05160>
- Girdhar et al. (2023). *ImageBind: One Embedding Space To Bind Them All*. CVPR. <https://arxiv.org/abs/2305.05665>
- Elizalde et al. (2022). *CLAP: Learning Audio Concepts From Natural Language Supervision*. <https://arxiv.org/abs/2206.04769>
- Xiao et al. (2025). *MIEB: Massive Image Embedding Benchmark*. <https://arxiv.org/abs/2504.10471>
- Lee et al. (2025). *Gemini Embedding: Generalizable Embeddings from Gemini*. <https://arxiv.org/abs/2503.07891>
- Khattab & Zaharia (2020). *ColBERT*. SIGIR. <https://arxiv.org/abs/2004.12832> · Dhulipala et al. (2024). *MUVERA: Multi-Vector Retrieval via Fixed Dimensional Encodings*. <https://arxiv.org/abs/2405.19504>
- Anuncio de disponibilidad general de Gemini Embedding 2 (Google, abr-2026): <https://blog.google/innovation-and-ai/models-and-research/gemini-models/gemini-embedding-2-generally-available/> *(fechas y límites contrastados con cobertura de prensa; confirma en la documentación oficial)*.
- Paquete `colpali-engine` (PyPI) <https://pypi.org/project/colpali-engine/>.
- Material del curso: módulos 08 (tokens multimodales), 11, 13, 14, 15 (`poliglota_eval.py`), 18.